In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

## Load file

In [2]:
X_Uncleaned = pd.read_csv("../X.csv")
y_Uncleaned = pd.read_csv("../y.csv")

X = X_Uncleaned.iloc[:, 1:]  # remove index
y = y_Uncleaned.iloc[:, 1:]  # remove index

# display(y.head())

In [3]:
loadPickleFile = pd.read_pickle("../xgb_model_withgpa.pkl")
# print(loadPickleFile)

### Preprocess X and Y

In [4]:
y_binary = np.where(y >= 10, 1, 0)


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y_binary, random_state=42
)

y_test_binary = np.where(y_test >= 10, 1, 0)


### Build Model

In [5]:
# Extract the hyperparams
params = (
    loadPickleFile.get_params()
)  # --> Used to get paramters of a fresh model (not trained yet)

### Baseline Training

In [6]:
model_baseline = XGBRegressor(**params)
model_baseline.fit(X_train, y_train)

y_pred_scores_baseline = model_baseline.predict(X_test)

### Baseline Model validation

In [7]:
# RMSE
rmse_baseline = np.sqrt(mean_squared_error(y_test, y_pred_scores_baseline))
print("RMSE:", rmse_baseline)

# MAE
mae_baseline = mean_absolute_error(y_test, y_pred_scores_baseline)
print("MAE:", mae_baseline)

# R-squared
r2_baseline = r2_score(y_test, y_pred_scores_baseline)
print("R² Score:", r2_baseline)

RMSE: 1.8003777240101417
MAE: 1.4757369756698608
R² Score: 0.9037301540374756


## Add noise on Top 3 important variables

In [8]:
print("sadornot distinct values:", X_train['sadornot'].unique())
print("experiences distinct values:", X_train['experiences'].unique())
print("gpa_all distinct values:", X_train['gpa_all'].unique())

sadornot distinct values: [1. 2.]
experiences distinct values: [2. 5. 4. 3. 1.]
gpa_all distinct values: [3.373 3.476 2.815 3.947 3.519 3.705 3.029 2.4   3.79  3.667 3.719 3.625
 3.474 3.293 3.245 3.826 3.505]


In [9]:
# add noise to variables: sadornot
# Randomly add 1, subtract 1, or make no change for each sample.
noisy_X_train = X_train.copy()

sadornot_noise = np.random.choice([-1, 0, 1], size=noisy_X_train.shape[0])

noisy_X_train['sadornot'] = noisy_X_train['sadornot'] + sadornot_noise
noisy_X_train['sadornot'] = noisy_X_train['sadornot'].clip(lower=1, upper=2)

print(noisy_X_train['sadornot'].value_counts().sort_index())

sadornot
1.0    3820
2.0    3382
Name: count, dtype: int64


In [10]:
# add noise to variable: experiences
# apply normal distribution to choose the reasonable noise to add on the original experience column

experiences_noise = np.random.choice([-1, 0, 1], size=noisy_X_train.shape[0], p=[0.25, 0.5, 0.25])

noisy_X_train['experiences'] = noisy_X_train['experiences'] + experiences_noise
noisy_X_train['experiences'] = noisy_X_train['experiences'].clip(lower=1, upper=5)

print(noisy_X_train['experiences'].head(20))

2932    3.0
3445    5.0
5394    3.0
4079    3.0
2930    2.0
7641    2.0
1210    3.0
3669    3.0
5673    4.0
516     2.0
2934    1.0
7287    2.0
8570    3.0
8462    1.0
6052    1.0
3518    5.0
2988    3.0
1406    3.0
2942    2.0
3255    5.0
Name: experiences, dtype: float64


In [11]:
# add noise to variable: gpa_all
# gpa_all is continuous numerical variable
# apply normal distribution to choose the reasonable noise to add on the original sleep_hours column

gpa_noise = np.random.normal(0, 0.05, size=noisy_X_train.shape[0])
noisy_X_train['gpa_all'] = noisy_X_train['gpa_all'] + gpa_noise
print(noisy_X_train['gpa_all'].head(10))

2932    3.428460
3445    3.456919
5394    2.825067
4079    3.949388
2930    3.348964
7641    3.513430
1210    3.620784
3669    3.881180
5673    2.828654
516     3.113585
Name: gpa_all, dtype: float64


## re-train the model with Top3 variables + noise

In [12]:
model_noise_top3 = XGBRegressor(**params)
model_noise_top3.fit(noisy_X_train, y_train)

y_pred_noise_top3 = model_noise_top3.predict(X_test)

## validation

In [13]:
# RMSE
rmse_top3 = np.sqrt(mean_squared_error(y_test, y_pred_noise_top3))
print("RMSE:", rmse_top3)

# MAE
mae_top3 = mean_absolute_error(y_test, y_pred_noise_top3)
print("MAE:", mae_top3)

# R-squared
r2_top3 = r2_score(y_test, y_pred_noise_top3)
print("R² Score:", r2_top3)

RMSE: 1.9036178552021437
MAE: 1.5095044374465942
R² Score: 0.8923726677894592


## Add noise on Bottom 3 important variables

In [14]:
print("has_negative_text distinct values:", X_train['has_negative_text'].unique())
print("schedule distinct values:", X_train['schedule'].unique())
print("have distinct values:", X_train['have'].unique())

has_negative_text distinct values: [1. 0.]
schedule distinct values: [1. 2.]
have distinct values: [1. 2.]


In [15]:
less_relevent_noisy_X_train = X_train.copy()

binary_features = {
    'has_negative_text': (0, 1),
    'schedule': (1, 2),
    'have': (1, 2)
}

for feature, (min_val, max_val) in binary_features.items():
    noise = np.random.choice([-1, 0, 1], size=less_relevent_noisy_X_train.shape[0])
    less_relevent_noisy_X_train[feature] = less_relevent_noisy_X_train[feature] + noise
    less_relevent_noisy_X_train[feature] = less_relevent_noisy_X_train[feature].clip(lower=min_val, upper=max_val)

print(less_relevent_noisy_X_train[list(binary_features.keys())].head(10))

      has_negative_text  schedule  have
2932                1.0       2.0   1.0
3445                1.0       1.0   2.0
5394                1.0       2.0   1.0
4079                0.0       2.0   1.0
2930                1.0       1.0   2.0
7641                1.0       1.0   2.0
1210                1.0       1.0   1.0
3669                1.0       2.0   1.0
5673                0.0       2.0   1.0
516                 1.0       2.0   1.0


## re-train the model with bottom 3 variables + noise

In [ ]:
model_noise_bottom3 = XGBRegressor(**params)
model_noise_bottom3.fit(less_relevent_noisy_X_train, y_train)

y_pred_noise_bottom3 = model_noise_bottom3.predict(X_test)

## validation

In [18]:
# RMSE
rmse_bottom3 = np.sqrt(mean_squared_error(y_test, y_pred_noise_bottom3))
print("RMSE:", rmse_bottom3)

# MAE
mae_bottom3 = mean_absolute_error(y_test, y_pred_noise_bottom3)
print("MAE:", mae_bottom3)

# R-squared
r2_bottom3 = r2_score(y_test, y_pred_noise_bottom3)
print("R² Score:", r2_bottom3)

RMSE: 1.8015024722736386
MAE: 1.4751765727996826
R² Score: 0.9036098122596741
